# Train the question-generation model in Google Colab

This notebook fine-tunes `google/flan-t5-base` on the project's local SQuADv2 Parquet splits. Enable a Colab GPU before running the training cell: **Runtime > Change runtime type > T4 GPU**.

The default path reads the dataset from Google Drive. Set `USE_GOOGLE_DRIVE = False` to upload the two Parquet files directly into the temporary Colab runtime. The trained checkpoint is written to Drive when Drive mode is enabled.

In [ ]:
!pip -q install 'transformers>=4.45,<5' 'datasets>=2.20,<4' 'accelerate>=0.34,<2' 'pyarrow>=15,<24'

In [ ]:
from pathlib import Path
import shutil
import torch

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Automated-MCQ-Generator')
MODEL_NAME = 'google/flan-t5-base'
OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'models/question_generation/flan-t5-colab'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRAIN_PATH = DRIVE_PROJECT_ROOT / 'data/training/train-00000-of-00001.parquet'
    VALIDATION_PATH = DRIVE_PROJECT_ROOT / 'data/evaluation/validation-00000-of-00001.parquet'
else:
    from google.colab import files
    upload_dir = Path('/content/parquet_data')
    upload_dir.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    for filename in uploaded:
        shutil.move(filename, upload_dir / Path(filename).name)
    TRAIN_PATH = upload_dir / 'train-00000-of-00001.parquet'
    VALIDATION_PATH = upload_dir / 'validation-00000-of-00001.parquet'
    OUTPUT_DIR = Path('/content/flan-t5-colab')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Change the Colab runtime to a T4 GPU before training.')

print('Train:', TRAIN_PATH)
print('Validation:', VALIDATION_PATH)
print('Output:', OUTPUT_DIR)

In [ ]:
from datasets import Dataset, DatasetDict
import pyarrow as pa
import pyarrow.parquet as pq

def load_prepared_parquet(path: Path) -> Dataset:
    table = pq.read_table(path, columns=['id', 'context', 'question', 'answers'])
    records = []
    for row in table.to_pylist():
        answers = row.get('answers') or {}
        texts = answers.get('text') or []
        if not texts or not row.get('context') or not row.get('question'):
            continue
        context = ' '.join(str(row['context']).split())
        question = ' '.join(str(row['question']).split())
        answer = str(texts[0]).strip()
        if context and question and answer:
            records.append({
                'id': row.get('id', ''),
                'input_text': f'generate question: context: {context} answer: {answer}',
                'target_text': question,
            })
    if not records:
        raise ValueError(f'No answerable examples found in {path}')
    return Dataset(pa.Table.from_pylist(records))

if not TRAIN_PATH.is_file() or not VALIDATION_PATH.is_file():
    raise FileNotFoundError(
        'Dataset files were not found. Copy the repository into Google Drive at '
        f'{DRIVE_PROJECT_ROOT}'
    )

dataset = DatasetDict({
    'train': load_prepared_parquet(TRAIN_PATH),
    'validation': load_prepared_parquet(VALIDATION_PATH),
})
print(dataset)
print(dataset['train'][0])

## Training settings

The settings below are suitable for a Colab T4. Reduce `BATCH_SIZE` or set `MAX_TRAIN_EXAMPLES` for a smoke test. Set `MAX_TRAIN_EXAMPLES = None` for the full split.

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

EPOCHS = 3
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64
MAX_TRAIN_EXAMPLES = None
MAX_VALIDATION_EXAMPLES = None
SEED = 42

set_seed(SEED)
if MAX_TRAIN_EXAMPLES is not None:
    dataset['train'] = dataset['train'].select(range(min(MAX_TRAIN_EXAMPLES, len(dataset['train']))))
if MAX_VALIDATION_EXAMPLES is not None:
    dataset['validation'] = dataset['validation'].select(range(min(MAX_VALIDATION_EXAMPLES, len(dataset['validation']))))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=MAX_TARGET_LENGTH, truncation=True)
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc='Tokenizing question-generation data',
)

use_fp16 = torch.cuda.is_available()
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    optim='adamw_torch',
    predict_with_generate=False,
    report_to=[],
    load_best_model_at_end=True,
    save_total_limit=2,
    fp16=use_fp16,
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)
print(f'Train examples: {len(dataset["train"])}')
print(f'Validation examples: {len(dataset["validation"])}')
print(f'FP16 enabled: {use_fp16}')

In [ ]:
train_result = trainer.train()
validation_metrics = trainer.evaluate()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

import json
run_record = {
    'model_name': MODEL_NAME,
    'training_mode': 'google_colab',
    'train_file': str(TRAIN_PATH),
    'validation_file': str(VALIDATION_PATH),
    'train_examples': len(dataset['train']),
    'validation_examples': len(dataset['validation']),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'max_input_length': MAX_INPUT_LENGTH,
    'max_target_length': MAX_TARGET_LENGTH,
    'seed': SEED,
    'train_metrics': train_result.metrics,
    'validation_metrics': validation_metrics,
}
(OUTPUT_DIR / 'training_run.json').write_text(json.dumps(run_record, indent=2, default=str) + '\n')
print(json.dumps(run_record, indent=2, default=str))
print(f'Checkpoint saved to: {OUTPUT_DIR}')

## Quick generation check

This confirms that the saved checkpoint can generate a question from the same answer-aware format used by the project.

In [ ]:
sample = dataset['validation'][0]
inputs = tokenizer(sample['input_text'], return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH).to(model.device)
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4)
print('Input:', sample['input_text'])
print('Expected:', sample['target_text'])
print('Generated:', tokenizer.decode(generated_ids[0], skip_special_tokens=True))